In [101]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [102]:
# Load data
ratings_df = pd.read_csv("book_ratings.csv", delimiter=';', encoding='latin1')
books_df = pd.read_csv("books.csv", delimiter=';', encoding='latin1', on_bad_lines='skip')
users_df = pd.read_csv("users.csv", delimiter=';', encoding='latin1')

/var/folders/r1/l_8tpbc17fq5_j__fql0b70h0000gn/T/ipykernel_5598/2430909678.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df = pd.read_csv("books.csv", delimiter=';', encoding='latin1', on_bad_lines='skip')


In [103]:
ratings_df

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6
...,...,...,...
1149775,276704,1563526298,9
1149776,276706,0679447156,0
1149777,276709,0515107662,10
1149778,276721,0590442449,10


In [104]:
books_df

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...
...,...,...,...,...,...,...,...,...
271355,0440400988,There's a Bat in Bunk Five,Paula Danziger,1988,Random House Childrens Pub (Mm),http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...
271356,0525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...
271358,0192126040,Republic (World's Classics),Plato,1996,Oxford University Press,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...


In [105]:
users_df

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN
...,...,...,...
278853,278854,"portland, oregon, usa",NaN
278854,278855,"tacoma, washington, united kingdom",50.0
278855,278856,"brampton, ontario, canada",NaN
278856,278857,"knoxville, tennessee, usa",NaN


In [106]:
# Step 4: Check columns of dataframes
print(ratings_df.columns)
print(books_df.columns)
print(users_df.columns)


Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='object')
Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')
Index(['User-ID', 'Location', 'Age'], dtype='object')


In [107]:
print(users_df.shape)
print(books_df.shape)
print(ratings_df.shape)

(278858, 3)
(271360, 8)
(1149780, 3)


In [108]:
data = pd.merge(ratings_df, users_df, on='User-ID', how='inner')
data = pd.merge(data, books_df, on='ISBN', how='inner')

In [109]:
data.head()

,User-ID,ISBN,Book-Rating,Location,Age,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,276725,034545104X,0,"tyler, texas, usa",NaN,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
1,2313,034545104X,5,"cincinnati, ohio, usa",23.0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
2,6543,034545104X,0,"strafford, missouri, usa",34.0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
3,8680,034545104X,5,"st. charles county, missouri, usa",2.0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
4,10314,034545104X,9,"beaverton, oregon, usa",NaN,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...


In [110]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031136 entries, 0 to 1031135
Data columns (total 12 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   User-ID              1031136 non-null  int64  
 1   ISBN                 1031136 non-null  object 
 2   Book-Rating          1031136 non-null  int64  
 3   Location             1031136 non-null  object 
 4   Age                  753301 non-null   float64
 5   Book-Title           1031136 non-null  object 
 6   Book-Author          1031134 non-null  object 
 7   Year-Of-Publication  1031136 non-null  object 
 8   Publisher            1031134 non-null  object 
 9   Image-URL-S          1031136 non-null  object 
 10  Image-URL-M          1031136 non-null  object 
 11  Image-URL-L          1031132 non-null  object 
dtypes: float64(1), int64(2), object(9)
memory usage: 94.4+ MB


In [111]:
data.drop(['Image-URL-S', 'Image-URL-M', 'Image-URL-L'], axis=1, inplace=True)


In [112]:
data.head()

,User-ID,ISBN,Book-Rating,Location,Age,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,276725,034545104X,0,"tyler, texas, usa",NaN,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books
1,2313,034545104X,5,"cincinnati, ohio, usa",23.0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books
2,6543,034545104X,0,"strafford, missouri, usa",34.0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books
3,8680,034545104X,5,"st. charles county, missouri, usa",2.0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books
4,10314,034545104X,9,"beaverton, oregon, usa",NaN,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books


In [113]:
data.isnull().sum()

User-ID                     0
ISBN                        0
Book-Rating                 0
Location                    0
Age                    277835
Book-Title                  0
Book-Author                 2
Year-Of-Publication         0
Publisher                   2
dtype: int64

In [114]:
num_users = data['User-ID'].nunique()
num_books = data['ISBN'].nunique()
print(f"Number of users: {num_users}")
print(f"Number of books: {num_books}")


Number of users: 92106
Number of books: 270151


In [115]:
median_age = data['Age'].median()
data.loc[data['Age'] > 100, 'Age'] = median_age

In [116]:
# Remove missing values in 'Book-Author' and 'Publisher' columns
data.dropna(subset=['Book-Author', 'Publisher'], inplace=True)

# Median impute the 'Age' column
data['Age'].fillna(data['Age'].median(), inplace=True)


In [117]:
data.isnull().sum()

User-ID                0
ISBN                   0
Book-Rating            0
Location               0
Age                    0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
Publisher              0
dtype: int64

In [118]:
data.shape

(1031132, 9)

In [119]:
data['Year-Of-Publication'] = pd.to_datetime(data['Year-Of-Publication'], errors='coerce').dt.year.fillna(0).astype(int)


In [120]:
# Check the length of the columns after splitting 'Location' column
location_split = data['Location'].str.split(', ', expand=True)
print(location_split.head())


                    0         1    2     3     4     5     6
0               tyler     texas  usa  None  None  None  None
1          cincinnati      ohio  usa  None  None  None  None
2           strafford  missouri  usa  None  None  None  None
3  st. charles county  missouri  usa  None  None  None  None
4           beaverton    oregon  usa  None  None  None  None


In [21]:
data[['City', 'State', 'Country']] = data['Location'].str.split(', ', expand=True, n=2).fillna('')

In [22]:
data.head()

,User-ID,ISBN,Book-Rating,Location,Age,Book-Title,Book-Author,Year-Of-Publication,Publisher,City,State,Country
0,276725,034545104X,0,"tyler, texas, usa",35.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,tyler,texas,usa
1,2313,034545104X,5,"cincinnati, ohio, usa",23.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,cincinnati,ohio,usa
2,6543,034545104X,0,"strafford, missouri, usa",34.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,strafford,missouri,usa
3,8680,034545104X,5,"st. charles county, missouri, usa",2.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,st. charles county,missouri,usa
4,10314,034545104X,9,"beaverton, oregon, usa",35.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,beaverton,oregon,usa


In [23]:
# Split 'Location' column into separate columns for city, state, and country
split_columns = data['Location'].str.split(', ', expand=True)

# Extract 'City', 'State', and 'Country' columns
data['City'] = split_columns[0]
data['State'] = split_columns[1]
data['Country'] = split_columns[2]

# Drop the original 'Location' column and any extra columns with 'None' values
data.drop(['Location'], axis=1, inplace=True)


In [24]:
data.head()

,User-ID,ISBN,Book-Rating,Age,Book-Title,Book-Author,Year-Of-Publication,Publisher,City,State,Country
0,276725,034545104X,0,35.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,tyler,texas,usa
1,2313,034545104X,5,23.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,cincinnati,ohio,usa
2,6543,034545104X,0,34.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,strafford,missouri,usa
3,8680,034545104X,5,2.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,st. charles county,missouri,usa
4,10314,034545104X,9,35.0,Flesh Tones: A Novel,M. J. Rose,1970,Ballantine Books,beaverton,oregon,usa


In [25]:
# Get the shape of the DataFrame new_df
print(data.shape)


(1031132, 11)


In [26]:
data.isnull().sum()

User-ID                    0
ISBN                       0
Book-Rating                0
Age                        0
Book-Title                 0
Book-Author                0
Year-Of-Publication        0
Publisher                  0
City                       0
State                      0
Country                24195
dtype: int64

In [27]:
from sklearn.preprocessing import LabelEncoder

# Features to be label encoded
features_to_encode = [ 'Book-Title', 'Book-Author', 'Publisher', 'City', 'State', 'Country']

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Initialize a dictionary to store the mapping of encoded values to original values
encoded_mapping = {}

# Apply label encoding to each feature and store the mapping
for feature in features_to_encode:
    data[feature + '_encoded'] = label_encoder.fit_transform(data[feature].astype(str))
    encoded_mapping[feature] = dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))

# Display the encoded features with corresponding index
encoded_features = data[[ 'Book-Title_encoded', 'Book-Author_encoded', 'Publisher_encoded', 'City_encoded', 'State_encoded', 'Country_encoded']]
encoded_features.columns = [f'{col}_encoded_{i}' for i, col in enumerate(encoded_features.columns)]
print(encoded_features)



         Book-Title_encoded_encoded_0  Book-Author_encoded_encoded_1  \
0                               67828                          61681   
1                               67828                          61681   
2                               67828                          61681   
3                               67828                          61681   
4                               67828                          61681   
...                               ...                            ...   
1031131                        123562                          24409   
1031132                         77085                          89212   
1031133                        222386                          21499   
1031134                          2960                          51694   
1031135                        139424                          76091   

         Publisher_encoded_encoded_2  City_encoded_encoded_3  \
0                               1388                   13340   
1      

In [28]:
data.columns

Index(['User-ID', 'ISBN', 'Book-Rating', 'Age', 'Book-Title', 'Book-Author',
       'Year-Of-Publication', 'Publisher', 'City', 'State', 'Country',
       'Book-Title_encoded', 'Book-Author_encoded', 'Publisher_encoded',
       'City_encoded', 'State_encoded', 'Country_encoded'],
      dtype='object')

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F 

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.modules.batchnorm import BatchNorm1d

class WideAndDeepModel(nn.Module):
    def __init__(self, embedding_sizes, n_continuous):
        super(WideAndDeepModel, self).__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(categories, size) for categories, size in embedding_sizes])
        n_emb = sum(e.embedding_dim for e in self.embeddings)
        self.n_emb, self.n_continuous = n_emb, n_continuous
        self.lin1 = nn.Linear(self.n_emb + self.n_continuous, 300)  # Increased units in hidden layers
        self.lin2 = nn.Linear(300, 200)  # Increased units in hidden layers
        self.lin3 = nn.Linear(200, 100)  # Increased units in hidden layers
        self.lin4 = nn.Linear(100, 50)
        self.lin5 = nn.Linear(50, 11)  # 11 output classes for Book-Rating
        self.emb_drop = nn.Dropout(0.6)
        self.drops = nn.Dropout(0.4)  # Increased dropout rate
        self.bn1 = BatchNorm1d(self.n_continuous)
        self.bn2 = BatchNorm1d(300)
        self.bn3 = BatchNorm1d(200)
        self.bn4 = BatchNorm1d(100)
    
    def forward(self, x_cat, x_continuous):
        x = [e(x_cat[:, i]) for i, e in enumerate(self.embeddings)]
        x = torch.cat(x, 1)
        x = self.emb_drop(x)
        x_continuous = self.bn1(x_continuous)
        x = torch.cat([x, x_continuous], 1)
        x = F.relu(self.bn2(self.lin1(x)))
        x = self.drops(x)
        x = F.relu(self.bn3(self.lin2(x)))
        x = self.drops(x)
        x = F.relu(self.bn4(self.lin3(x)))  # Added another hidden layer
        x = self.lin4(x)
        x = self.lin5(x)
        return x


In [31]:
# Training function
def train_model(model, criterion, optimizer, cat_train, cont_train, y_train, cat_test, cont_test, y_test, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()

        y_pred = model(cat_train, cont_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            y_val = model(cat_test, cont_test)
            val_loss = criterion(y_val, y_test)

        print(f"Epoch {epoch+1}/{n_epochs} - Loss: {loss.item()}, Val Loss: {val_loss.item()}")

In [32]:
from sklearn.model_selection import train_test_split
from sklearn import metrics, preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Split the data
train_data, test_data = train_test_split(data, test_size=0.2, random_state=46)

In [33]:
# Define model parameters
embedding_sizes = [(len(data[col].unique()), min(50, (len(data[col].unique())+1)//2)) for col in ['Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'City', 'State', 'Country']]
continuous_feature_age = 1


In [61]:
print(embedding_sizes)

[(241068, 50), (101586, 50), (93, 47), (16728, 50), (14769, 50), (2125, 50), (389, 50)]


In [62]:
# Assuming embedding_sizes and cat_columns are defined
num_embedding_layers = len(embedding_sizes)
num_categorical_columns = len(cat_columns)

print("Number of embedding layers:", num_embedding_layers)
print("Number of categorical columns:", num_categorical_columns)

if num_embedding_layers == num_categorical_columns:
    print("Number of embedding layers matches the number of categorical columns.")
else:
    print("Warning: Number of embedding layers does not match the number of categorical columns.")


Number of embedding layers: 7
Number of categorical columns: 7
Number of embedding layers matches the number of categorical columns.


In [34]:
model = WideAndDeepModel(embedding_sizes, continuous_feature_age)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [35]:
# Split the data
train_data, test_data = train_test_split(data, test_size=0.2, random_state=46)

# Separate features and target variable
cat_columns = [ 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'City', 'State', 'Country']
cont_columns = ['Age']  # Assuming 'Age' is the only continuous feature
target_column = 'Book-Rating'

cat_train = train_data[cat_columns]
cont_train = train_data[cont_columns]
y_train = train_data[target_column]

cat_test = test_data[cat_columns]
cont_test = test_data[cont_columns]
y_test = test_data[target_column]

In [64]:
# Convert continuous data to float
cont_train = cont_train.astype(float)
cont_test = cont_test.astype(float)

# Convert target variable to integers
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Convert data to tensors
cat_train_tensor = torch.tensor(cat_train.values, dtype=torch.long)
cont_train_tensor = torch.tensor(cont_train.values, dtype=torch.float)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

cat_test_tensor = torch.tensor(cat_test.values, dtype=torch.long)
cont_test_tensor = torch.tensor(cont_test.values, dtype=torch.float)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)


In [65]:
# Train the model
n_epochs = 30
train_model(model, criterion, optimizer, cat_train_tensor, cont_train_tensor, y_train_tensor, cat_test_tensor, cont_test_tensor, y_test_tensor, n_epochs)

# Use the model for predictions
with torch.no_grad():
    final_preds = model(cat_test_tensor, cont_test_tensor)
    final_preds = torch.argmax(final_preds, dim=1)
    accuracy = (final_preds == y_test_tensor).float().mean()

print(f"Prediction Accuracy: {accuracy*100:.2f}%")

Epoch 1/30 - Loss: 1.3248850107192993, Val Loss: 1.4150370359420776
Epoch 2/30 - Loss: 1.3221818208694458, Val Loss: 1.412954330444336
Epoch 3/30 - Loss: 1.3199745416641235, Val Loss: 1.41178560256958
Epoch 4/30 - Loss: 1.3180900812149048, Val Loss: 1.4121400117874146
Epoch 5/30 - Loss: 1.3172348737716675, Val Loss: 1.4140677452087402
Epoch 6/30 - Loss: 1.3151317834854126, Val Loss: 1.417396068572998
Epoch 7/30 - Loss: 1.313259243965149, Val Loss: 1.4215797185897827
Epoch 8/30 - Loss: 1.310957670211792, Val Loss: 1.425859808921814
Epoch 9/30 - Loss: 1.3078677654266357, Val Loss: 1.42960786819458
Epoch 10/30 - Loss: 1.3068777322769165, Val Loss: 1.431978464126587
Epoch 11/30 - Loss: 1.3050732612609863, Val Loss: 1.4326746463775635
Epoch 12/30 - Loss: 1.3030717372894287, Val Loss: 1.4321736097335815
Epoch 13/30 - Loss: 1.3004897832870483, Val Loss: 1.4314279556274414
Epoch 14/30 - Loss: 1.298066258430481, Val Loss: 1.431525707244873
Epoch 15/30 - Loss: 1.2964510917663574, Val Loss: 1.432